ДЗ 14.1. Виняток користувача

1. Модифікувала клас Group так, щоб при спробі додати більше 10 студентів генерувався власний виняток, який слід перехопити та обробити поза межами класу.

In [12]:
class Human:

    def __init__(self, gender, age, first_name, last_name):
        self.gender = gender
        self.age = age
        self.first_name = first_name
        self.last_name = last_name

    def __str__(self):
        return f'{self.gender} {self.age} {self.first_name} {self.last_name}'


class Student(Human):

    def __init__(self, gender, age, first_name, last_name, record_book):
        super().__init__(gender, age, first_name, last_name)
        self.record_book = record_book

    def __str__(self):
        return f'{super().__str__()} {self.record_book}'

    def __eq__(self, other):
        if isinstance(other, Student):
            return self.record_book == other.record_book
        return False

    def __hash__(self):
        return hash(self.record_book)


# Власний виняток
class GroupLimitError(Exception):
    pass


class Group:

    def __init__(self, number):
        self.number = number
        self.group = set()

    def add_student(self, student):
        # якщо студент уже є в групі — просто нічого не робимо
        if student in self.group:
            return

        # ліміт 10 студентів
        if len(self.group) >= 10:
            raise GroupLimitError("Не можна додати більше 10 студентів у групу.")

        self.group.add(student)

    def find_student(self, last_name):
        last_name = last_name.strip().lower()
        for student in self.group:
            if student.last_name.strip().lower() == last_name:
                return student
        return None

    def delete_student(self, last_name):
        student = self.find_student(last_name)
        if student is not None:
            self.group.remove(student)

    def __str__(self):
        all_students = ''
        for student in self.group:
            all_students += str(student) + '\n'
        return f'Number:{self.number}\n{all_students}'


st1 = Student('Male', 30, 'Steve', 'Jobs', 'AN142')
st2 = Student('Female', 25, 'Liza', 'Taylor', 'AN145')
gr = Group('PD1')

gr.add_student(st1)
gr.add_student(st2)

print(gr)

assert str(gr.find_student('Jobs')) == str(st1), 'Test1'
assert gr.find_student('Jobs2') is None, 'Test2'
assert isinstance(gr.find_student('Jobs'), Student) is True, 'Метод пошуку повинен повертати екземпляр'
assert str(gr.find_student('  jobs  ')) == str(st1), 'Test3 (case-insensitive + spaces)'

gr.delete_student('taylor')
print(gr)

gr.delete_student('Taylor')  # No error!


#  Демонстрація винятку: пробуємо додати 11-го
try:
    for i in range(3, 13):  # додамо ще 10 студентів (разом стане 11+)
        gr.add_student(Student('Male', 20, f'Name{i}', f'Last{i}', f'RB{i}'))
except GroupLimitError as e:
    print("Помилка додавання:", e)

Number:PD1
Female 25 Liza Taylor AN145
Male 30 Steve Jobs AN142

Number:PD1
Male 30 Steve Jobs AN142

Помилка додавання: Не можна додати більше 10 студентів у групу.


2. Код реалізує систему керування студентською групою з можливістю додавання, пошуку та видалення студентів, при цьому обмежує групу максимум 10 студентами та генерує користувацький виняток при спробі додати одинадцятого.

In [11]:
class Human:
    def __init__(self, gender, age, first_name, last_name):
        self.gender = gender
        self.age = age
        self.first_name = first_name
        self.last_name = last_name

    def __str__(self):
        return f'{self.gender} {self.age} {self.first_name} {self.last_name}'


class Student(Human):
    def __init__(self, gender, age, first_name, last_name, record_book):
        super().__init__(gender, age, first_name, last_name)
        self.record_book = record_book

    def __str__(self):
        return f'{super().__str__()} {self.record_book}'


# Власний виняток
class GroupLimitError(Exception):
    pass


class Group:
    def __init__(self, number):
        self.number = number
        self.group = []  # список студентів

    def add_student(self, student):
        # перевірка дублікатів
        for s in self.group:
            if s.record_book == student.record_book:
                return

        # перевірка ліміту
        if len(self.group) >= 10:
            raise GroupLimitError("Group limit exceeded (max 10 students)")

        self.group.append(student)

    def find_student(self, last_name):
        last_name = last_name.strip().lower()
        for student in self.group:
            if student.last_name.strip().lower() == last_name:
                return student
        return None

    def delete_student(self, last_name):
        student = self.find_student(last_name)
        if student is not None:
            self.group.remove(student)

    def __str__(self):
        all_students = ''
        for student in self.group:
            all_students += str(student) + '\n'
        return f'Number:{self.number}\n{all_students}'


st1 = Student('Male', 30, 'Steve', 'Jobs', 'AN142')
st2 = Student('Female', 25, 'Liza', 'Taylor', 'AN145')
gr = Group('PD1')

gr.add_student(st1)
gr.add_student(st2)

print(gr)

assert str(gr.find_student('Jobs')) == str(st1), 'Test1'
assert gr.find_student('Jobs2') is None, 'Test2'
assert isinstance(gr.find_student('Jobs'), Student) is True, 'Метод пошуку повинен повертати екземпляр'

gr.delete_student('Taylor')
print(gr)

gr.delete_student('Taylor')  # No error!


# Перехоплення винятку поза класом
try:
    for i in range(3, 13):  # намагаємось додати до 11+
        gr.add_student(Student('Male', 20, f'Name{i}', f'Last{i}', f'RB{i}'))
except GroupLimitError as e:
    print("Exception caught:", e)

Number:PD1
Male 30 Steve Jobs AN142
Female 25 Liza Taylor AN145

Number:PD1
Male 30 Steve Jobs AN142

Exception caught: Group limit exceeded (max 10 students)


3. Цей варіант реалізує керування групою студентів із використанням dataclass, додає обмеження у 10 студентів через користувацький виняток і демонструє його перехоплення поза межами класу.

In [9]:
from dataclasses import dataclass


@dataclass
class Human:
    gender: str
    age: int
    first_name: str
    last_name: str

    def __str__(self):
        return f'{self.gender} {self.age} {self.first_name} {self.last_name}'


@dataclass
class Student(Human):
    record_book: str

    def __str__(self):
        return f'{super().__str__()} {self.record_book}'


# власний виняток
class GroupLimitError(Exception):
    pass


class Group:
    def __init__(self, number):
        self.number = number
        self.group = []  # список студентів

    def add_student(self, student: Student):
        # захист від дублікатів
        if any(s.record_book == student.record_book for s in self.group):
            return

        # перевірка ліміту
        if len(self.group) >= 10:
            raise GroupLimitError("Cannot add more than 10 students")

        self.group.append(student)

    def find_student(self, last_name: str):
        last_name = last_name.strip().lower()
        for student in self.group:
            if student.last_name.strip().lower() == last_name:
                return student
        return None

    def delete_student(self, last_name: str):
        student = self.find_student(last_name)
        if student is not None:
            self.group.remove(student)

    def __str__(self):
        all_students = ''
        for student in self.group:
            all_students += str(student) + '\n'
        return f'Number:{self.number}\n{all_students}'


st1 = Student('Male', 30, 'Steve', 'Jobs', 'AN142')
st2 = Student('Female', 25, 'Liza', 'Taylor', 'AN145')

gr = Group('PD1')
gr.add_student(st1)
gr.add_student(st2)

print(gr)

assert str(gr.find_student('Jobs')) == str(st1), 'Test1'
assert gr.find_student('Jobs2') is None, 'Test2'
assert isinstance(gr.find_student('Jobs'), Student) is True

gr.delete_student('Taylor')
print(gr)

gr.delete_student('Taylor')  # No error!


# демонстрація винятку (11-й студент)
try:
    for i in range(3, 13):
        gr.add_student(Student('Male', 20, f'Name{i}', f'Last{i}', f'RB{i}'))
except GroupLimitError as e:
    print("Exception caught:", e)

Number:PD1
Male 30 Steve Jobs AN142
Female 25 Liza Taylor AN145

Number:PD1
Male 30 Steve Jobs AN142

Exception caught: Cannot add more than 10 students
